In [58]:
import os
import streamlit as st
from dotenv import load_dotenv
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate

from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI

In [59]:
#loading api key from .env file
load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

if not GOOGLE_API_KEY:
    raise ValueError('GOOGLE_API_KEY is not PRESENT IN MY .env FILE')

In [60]:
#pdf to text
def read_pdfs(pdf_files):
    all_text= ""
    for pdf in pdf_files:
        reader= PdfReader(pdf)
        for page in reader.pages:
            text=page.extract_text()
            if text:
                all_text += text
    return all_text


In [61]:
read_pdfs(['attention.gemini.pdf'])

'Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurrence and convolutions\nentirely. Experiments on two machine translation tasks show these models to\nbe superior in quality while being more parallelizable and requiring signiﬁcantly

In [62]:
text= read_pdfs(['attention.gemini.pdf']) #we are reading the pdf file and storing the text in a variable called text

In [63]:
print(text) # we can print the text to see if it is being read correctly from the pdf file

Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring signiﬁcantly
less time to train. Our model a

In [64]:
#chunks trying to split the text into smaller chunks for better processing
#for splitting the text into chunks
def split_text(text):
    splitter= RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=300)
    return splitter.split_text(text)


In [65]:
split_text(text)

['Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best',
 'aidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose

In [66]:
spiltted_text= split_text(text) #we are splitting the text into smaller chunks and storing it in a variable called splitted_text

In [67]:
spiltted_text[0] #we can print the first chunk to see if it is being split correctly

'Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best'

In [68]:
#retriever to retrieve the relevant chunks from the splitted text
chunks=split_text(text)

In [69]:
chunks[0]

'Attention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗†\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best'

In [70]:
chunks[1]

'aidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,\nbased solely on attention mechanisms, dispensing with recurrence and convolutions\nentirely. Experiments on two machine translation tasks show these models to'

In [71]:
def create_embeddings():
    return HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

In [72]:
#building the vector database to store the embeddings of the chunks
def build_vector_store(chunks):
    embeddings= create_embeddings()
    documents= [Document(page_content=chunk) for chunk in chunks]
    vector_store= FAISS.from_documents(documents, embeddings)
    vector_store.save_local("faiss_index")

In [73]:
build_vector_store(chunks) #we are building the vector store and storing it in a folder called faiss_index

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2823.82it/s]


In [74]:
#loading the vector store to retrieve the relevant chunks from the splitted text
def load_vector_store():
    embeddings= create_embeddings()
    return FAISS.load_local("faiss_index", embeddings,allow_dangerous_deserialization=True)

In [75]:
#retiving the chunks or embeddings from the vector store
def retrieve_chunks(question):
    vector_store= load_vector_store()
    return vector_store.similarity_search(question,k=10)

In [76]:
docs= retrieve_chunks("explain self attention in transformers?") #we are retrieving the relevant chunks from the vector store based on the question

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5766.02it/s]


In [77]:
docs

[Document(id='c556dbca-d1c6-4951-81b1-0b9640329079', metadata={}, page_content='in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes\nit more difﬁcult to learn dependencies between distant positions [ 11]. In the Transformer this is\nreduced to a constant number of operations, albeit at the cost of reduced effective resolution due\nto averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as\ndescribed in section 3.2.\nSelf-attention, sometimes called intra-attention is an attention mechanism relating different positions'),
 Document(id='075d0157-6d51-42aa-8f30-d64082a9a89f', metadata={}, page_content='entirely on self-attention to compute representations of its input and output without using sequence-\naligned RNNs or convolution. In the following sections, we will describe the Transformer, motivate\nself-attention and discuss its advantages over models such as [14, 15] and [8].\n3 Model Architecture\nMo

In [78]:
for d in docs:
    print(d.page_content) #we can print the retrieved chunks to see if they are relevant to the question

in the distance between positions, linearly for ConvS2S and logarithmically for ByteNet. This makes
it more difﬁcult to learn dependencies between distant positions [ 11]. In the Transformer this is
reduced to a constant number of operations, albeit at the cost of reduced effective resolution due
to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as
described in section 3.2.
Self-attention, sometimes called intra-attention is an attention mechanism relating different positions
entirely on self-attention to compute representations of its input and output without using sequence-
aligned RNNs or convolution. In the following sections, we will describe the Transformer, motivate
self-attention and discuss its advantages over models such as [14, 15] and [8].
3 Model Architecture
Most competitive neural sequence transduction models have an encoder-decoder structure [5, 2, 29].
Here, the encoder maps an input sequence of symbol representations (x1,...,

In [79]:
#generation part to generate the answer based on the retrieved chunks and the question
prompt = PromptTemplate(
    template="""
Answer the question using ONLY the context.

If the answer is not present, say:
"THE ANSWER IS NOT AVAILABLE IN THE PROVIDED CONTEXT."

Explain at Class 10 level.
Answer in bullet points.

Context:
{context}

Question:
{question}
""",
    input_variables=["context", "question"]
)

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0
)

chain = prompt | llm


def answer_question(question):

    docs = retrieve_chunks(question)

    context = "\n\n".join(doc.page_content for doc in docs)

    response = chain.invoke({
        "context": context,
        "question": question
    })

    return response.content


In [80]:
answer_question("what is multihead attention")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9456.97it/s]
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'type': 'text',
  'text': 'Based on the provided text, here is an explanation of Multi-Head Attention:\n\n*   **Definition:** Multi-Head Attention is a mechanism that allows a model to look at information from different "representation subspaces" at different positions simultaneously.\n*   **How it works:** Instead of using a single attention mechanism, the model runs several attention layers (called "heads") in parallel.\n*   **The Process:** \n    *   The model performs the attention function on queries, keys, and values in parallel.\n    *   The outputs from these different heads are then concatenated (joined together).\n    *   Finally, these concatenated values are projected to produce the final output.\n*   **Why it is used:** \n    *   A single attention head tends to "average" attention-weighted positions, which can inhibit the model\'s performance. Multi-Head Attention counteracts this averaging effect.\n    *   It allows the model to focus on different types of information 

In [85]:
#slicing the list of dictionary based on the key 'text' to get the value of the key 'text' from the list of dictionaries
#generation part to generate the answer based on the retrieved chunks and the question
prompt = PromptTemplate(
    template="""
Answer the question using ONLY the context.

If the answer is not present, say:
"THE ANSWER IS NOT AVAILABLE IN THE PROVIDED CONTEXT."

Explain at Class 10 level.
Answer in bullet points.

Context:
{context}

Question:
{question}
""",
    input_variables=["context", "question"]
)

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0
)

chain = prompt | llm


def answer_question(question):

    docs = retrieve_chunks(question)

    context = "\n\n".join(doc.page_content for doc in docs)

    response = chain.invoke({
        "context": context,
        "question": question
    })

    return response.content[0]['text']  # Slicing the list of dictionaries to get the value of the key 'text'






In [88]:
answer_question("what is multihead attention")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1741.00it/s]


'Based on the provided text, here is an explanation of Multi-Head Attention:\n\n*   **Definition:** Multi-Head Attention is a mechanism that allows a model to look at information from different "representation subspaces" at different positions simultaneously.\n*   **How it works:** Instead of using a single attention mechanism, the model runs several attention layers (called "heads") in parallel.\n*   **The Process:** \n    *   The model performs the attention function on queries, keys, and values in parallel.\n    *   The outputs from these different heads are then concatenated (joined together).\n    *   Finally, these concatenated values are projected to produce the final output.\n*   **Why it is used:** \n    *   A single attention head tends to "average" attention-weighted positions, which can inhibit the model\'s performance. Multi-Head Attention counteracts this averaging effect.\n    *   It allows the model to focus on different types of information at the same time.\n*   **Eff